# Etapa 2 — Análisis del conjunto de datos

**Modelo de Machine Learning para el cribado técnico de reservorios candidatos a waterflooding**

Proyecto de titulación — Maestría en Petróleos, ESPOL
En cooperación con EP Petroecuador (Gerencia de Activo Auca)

---

Este cuaderno corresponde a la **Etapa 2** del flujo de trabajo metodológico y cubre sus cuatro casillas:

| Paso | Contenido |
|---|---|
| 2.1 | Estadísticas descriptivas y distribución de las variables |
| 2.2 | Análisis de correlación |
| 2.3 | Evaluación de balance de clases |
| 2.4 | Verificación de consistencia física del dataset |

**Propósito.** Auditar el conjunto de 5 000 escenarios generado en la Etapa 1 *antes* de
transformarlo o entrenar cualquier modelo. Aquí no se modifica nada: toda limpieza, escalado y
división en entrenamiento y prueba corresponde a la Etapa 3.

Las preguntas concretas que esta etapa debe responder son cuatro: si las distribuciones cubren el
espacio de interés, si hay variables que dupliquen información, si el balance de clases obliga a
remuestrear, y si el archivo contiene algún valor físicamente imposible.

## 1. Configuración del entorno

In [ ]:
import os, sys, subprocess

REPO_URL = "https://github.com/phabelog/waterflooding-ml-screening.git"
REPO_DIR = "waterflooding-ml-screening"

if os.path.isdir("src/analysis"):
    REPO_ROOT = "."
else:
    if not os.path.isdir(REPO_DIR):
        subprocess.run(["git", "clone", "-q", REPO_URL], check=True)
    REPO_ROOT = REPO_DIR

sys.path.insert(0, os.path.join(REPO_ROOT, "src", "analysis"))
sys.path.insert(0, os.path.join(REPO_ROOT, "src", "dataset_generation"))
print("Repositorio listo en:", os.path.abspath(REPO_ROOT))

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import data_analysis as da
from generate_dataset import FEATURE_COLUMNS, DERIVED_COLUMNS

plt.rcParams.update({"figure.dpi": 110, "font.size": 10})
AZUL, ROJO = "#1f4e79", "#c0392b"

ruta = os.path.join(REPO_ROOT, "data", "synthetic", "synthetic_dataset.csv")
df = pd.read_csv(ruta)

print(f"Escenarios: {len(df)}")
print(f"Variables de entrada del modelo: {len(FEATURE_COLUMNS)}")
print(f"Caracteristicas derivadas (no se entrenan): {len(DERIVED_COLUMNS)}")

### Variables de entrada frente a características derivadas

Conviene recordar la separación establecida en la Etapa 1, porque condiciona todo lo que sigue:
el modelo se entrena **solo** con las variables crudas del reservorio. Las salidas del framework
físico se conservan en el archivo para trazabilidad, pero quedan fuera del entrenamiento; de lo
contrario el clasificador reconstruiría la regla de etiquetado en lugar de aprender la física.

In [ ]:
print("ENTRADAS DEL MODELO:")
for c in FEATURE_COLUMNS:
    print("   ", c)
print("\nDERIVADAS (excluidas del entrenamiento):")
for c in DERIVED_COLUMNS:
    print("   ", c)

---
## 2. Paso 2.1 — Estadísticas descriptivas y distribución de las variables

In [ ]:
desc = da.descriptive_statistics(df)
desc.round(3)

La asimetría permite identificar variables que convendrá transformar en la Etapa 3. Se espera
un valor elevado en la permeabilidad, ya que se muestreó en escala logarítmica para cubrir tres
órdenes de magnitud; el resto de las variables debería mostrar distribuciones aproximadamente
uniformes, que es como fueron definidos sus rangos.

In [ ]:
variables = ["k_md", "phi", "h_net_ft", "api", "muo_cp", "Vdp",
             "Swi", "Sor", "So_current"]

fig, axes = plt.subplots(3, 3, figsize=(12, 8.5))
for ax, col in zip(axes.ravel(), variables):
    ax.hist(df[df.label == 0][col], bins=35, alpha=0.6, color=ROJO, label="No apto")
    ax.hist(df[df.label == 1][col], bins=35, alpha=0.6, color=AZUL, label="Apto")
    ax.set_xlabel(col); ax.set_ylabel("Frecuencia")
    if col in ("k_md", "muo_cp"):
        ax.set_xscale("log")
axes.ravel()[0].legend(fontsize=8)
fig.suptitle("Distribución de las variables de entrada por clase", fontsize=12)
plt.tight_layout(); plt.show()

### Poder discriminante individual

La **d de Cohen** mide, en desviaciones estándar, cuánto se separan las medias de ambas clases
para cada variable. Sirve para anticipar qué variables pesarán más en el modelo, pero no
sustituye al análisis de importancia de la Etapa 5: una variable con d pequeña puede resultar
decisiva al combinarse con otras.

In [ ]:
por_clase = da.distribution_by_class(df)
por_clase.round(3)

In [ ]:
top = por_clase.head(8).iloc[::-1]
fig, ax = plt.subplots(figsize=(7, 4))
colores = [AZUL if v > 0 else ROJO for v in top["d_de_Cohen"]]
ax.barh(top.index, top["d_de_Cohen"], color=colores)
ax.axvline(0, color="black", lw=0.8)
ax.set_xlabel("d de Cohen  (positivo = mayor en los casos aptos)")
ax.set_title("Poder discriminante individual de cada variable")
ax.grid(alpha=0.3, axis="x")
plt.tight_layout(); plt.show()

---
## 3. Paso 2.2 — Análisis de correlación

Se emplea el coeficiente de Spearman porque varias relaciones del framework son monótonas pero
no lineales, y Pearson las subestimaría.

El análisis persigue dos objetivos distintos. El primero es detectar **información duplicada**
entre variables de entrada: dos variables casi perfectamente correlacionadas aportan lo mismo y
el modelo reparte la importancia entre ellas de forma arbitraria, lo que arruina la
interpretabilidad. El segundo es comprobar que **ninguna variable aislada determina la
etiqueta**, lo que indicaría un criterio de cribado demasiado simple para justificar un modelo de
aprendizaje automático.

In [ ]:
corr = da.correlation_matrix(df)

fig, ax = plt.subplots(figsize=(9, 7.5))
im = ax.imshow(corr, cmap="RdBu_r", vmin=-1, vmax=1)
ax.set_xticks(range(len(corr))); ax.set_xticklabels(corr.columns, rotation=90, fontsize=8)
ax.set_yticks(range(len(corr))); ax.set_yticklabels(corr.columns, fontsize=8)
for i in range(len(corr)):
    for j in range(len(corr)):
        v = corr.iloc[i, j]
        if abs(v) > 0.3 and i != j:
            ax.text(j, i, f"{v:.2f}", ha="center", va="center",
                    fontsize=7, color="white" if abs(v) > 0.6 else "black")
fig.colorbar(im, ax=ax, shrink=0.8, label="Spearman")
ax.set_title("Matriz de correlación entre variables de entrada")
plt.tight_layout(); plt.show()

In [ ]:
redundantes = da.redundant_pairs(df, threshold=0.85)
if len(redundantes):
    print("Pares redundantes detectados:")
    display(redundantes.round(3))
else:
    print("Sin pares de variables redundantes (|r| >= 0.85).")

**Nota metodológica.** La primera versión del conjunto de datos incluía la temperatura del
reservorio entre las variables de entrada. Este mismo análisis reveló una correlación de −0.999
con la viscosidad del agua, que se deriva de ella mediante una función determinista: eran, en la
práctica, la misma variable. Se excluyó la temperatura del entrenamiento y se conservó la
viscosidad, que es la que interviene directamente en la física del desplazamiento. La temperatura
permanece en el archivo para trazabilidad.

Las correlaciones moderadas que subsisten son deliberadas y provienen de las relaciones de
consistencia física impuestas en la Etapa 1: la viscosidad del petróleo depende de la gravedad
API, y la saturación actual de petróleo depende de la saturación de agua irreducible.

In [ ]:
corr_label = da.correlation_with_label(df)
corr_label.round(3)

In [ ]:
r_max = corr_label["r_con_etiqueta"].abs().max()
print(f"Correlacion individual maxima con la etiqueta: |r| = {r_max:.3f}")
print()
if r_max < 0.85:
    print("Ninguna variable aislada determina la etiqueta.")
    print("La frontera de decision es multivariable, lo que justifica el uso")
    print("de un modelo de aprendizaje automatico frente a un umbral simple.")
else:
    print("ADVERTENCIA: una variable domina la etiqueta; revisar el criterio.")

---
## 4. Paso 2.3 — Evaluación de balance de clases

In [ ]:
balance = da.class_balance(df)
for k, v in balance.items():
    print(f"{k:20s}: {v}")

El balance determina si será necesario aplicar SMOTE en la Etapa 3. Conviene recordar que, de
aplicarse, debe hacerse **únicamente sobre el conjunto de entrenamiento** y después de la
división: generar ejemplos sintéticos antes de separar filtraría información del conjunto de
prueba y produciría métricas optimistas.

In [ ]:
rechazos = da.rejection_breakdown(df)

fig, axes = plt.subplots(1, 2, figsize=(12, 4.2))

conteos = [balance["n_no_apto"], balance["n_apto"]]
axes[0].bar(["No apto", "Apto"], conteos, color=[ROJO, AZUL], width=0.55)
for i, v in enumerate(conteos):
    axes[0].text(i, v + 40, f"{v}\n({v/len(df)*100:.1f} %)", ha="center", fontsize=9)
axes[0].set_ylim(0, max(conteos) * 1.25)
axes[0].set_ylabel("Numero de escenarios")
axes[0].set_title("Balance de clases")

orden = rechazos.iloc[::-1]
axes[1].barh(orden["criterio"], orden["% de no aptos"], color=ROJO, alpha=0.8)
axes[1].set_xlabel("% de los casos no aptos que incumplen el criterio")
axes[1].set_title("Criterios que motivan el rechazo")
axes[1].grid(alpha=0.3, axis="x")

plt.tight_layout(); plt.show()
rechazos.round(1)

Los porcentajes suman más de 100 % porque un mismo escenario puede incumplir varios criterios a
la vez. Lo relevante es que el rechazo no se concentra en un único criterio: si así fuera, el
problema se reduciría a un umbral y el modelo no tendría nada multivariable que aprender.

---
## 5. Paso 2.4 — Verificación de consistencia física del dataset

Esta verificación es **independiente** de la realizada en la Etapa 1. Allí se comprobó que el
framework calcula correctamente; aquí se audita el archivo resultante, buscando valores
imposibles que pudieran haberse introducido durante la generación o el guardado.

In [ ]:
consistencia = da.physical_consistency(df)
consistencia

In [ ]:
n_ok = int((consistencia["resultado"] == "PASA").sum())
print(f"{n_ok}/{len(consistencia)} verificaciones superadas")
if n_ok < len(consistencia):
    print("\nRevisar:")
    display(consistencia[consistencia["resultado"] == "FALLA"])

### Verificación de tendencias físicas

Más allá de que no existan valores imposibles, el conjunto agregado debe reproducir las
tendencias que la teoría predice. Se comprueban tres:

1. a mayor relación de movilidad, menor eficiencia de barrido areal;
2. a mayor heterogeneidad, menor eficiencia de barrido vertical;
3. a mayor recobro total, mayor proporción de casos aptos.

In [ ]:
mono = da.monotonicity_checks(df)

fig, axes = plt.subplots(1, 3, figsize=(13, 3.8))
titulos = [
    ("EA_vs_M", "Relacion de movilidad (deciles)", "$E_A$ media",
     "Barrido areal frente a movilidad"),
    ("EV_vs_Vdp", "$V_{dp}$ (deciles)", "$E_V$ media",
     "Barrido vertical frente a heterogeneidad"),
    ("apto_vs_RF", "Recobro total (deciles)", "Proporcion de aptos",
     "Aptitud frente a recobro"),
]

for ax, (clave, xlab, ylab, titulo) in zip(axes, titulos):
    serie = mono[clave]
    ax.plot(range(len(serie)), serie.values, "o-", color=AZUL, lw=2, ms=5)
    ax.set_xlabel(xlab); ax.set_ylabel(ylab)
    ax.set_title(titulo, fontsize=10); ax.grid(alpha=0.3)

plt.tight_layout(); plt.show()

for clave, _, _, titulo in titulos:
    v = mono[clave].values
    creciente = np.all(np.diff(v) >= -0.02)
    decreciente = np.all(np.diff(v) <= 0.02)
    estado = "monotona creciente" if creciente else ("monotona decreciente" if decreciente else "NO MONOTONA")
    print(f"{titulo:45s}: {estado}")

---
## 6. Conclusiones de la Etapa 2

**Distribuciones.** Las variables cubren los rangos definidos. La permeabilidad presenta la
asimetría esperada por su muestreo logarítmico y será candidata a transformación en la Etapa 3.
Las distribuciones por clase se solapan en todas las variables, lo cual es deseable: si alguna
separara limpiamente las clases, el problema no requeriría aprendizaje automático.

**Correlación.** No quedan pares de variables redundantes tras excluir la temperatura del
entrenamiento. Las correlaciones moderadas restantes provienen de las relaciones de consistencia
física impuestas deliberadamente. Ninguna variable aislada supera |r| = 0.41 con la etiqueta, lo
que confirma que la frontera de decisión es multivariable.

**Balance.** Cercano a la paridad, por lo que no se requiere remuestreo. SMOTE queda previsto en
la metodología únicamente como contingencia, aplicable solo al conjunto de entrenamiento.

**Consistencia física.** El conjunto supera la totalidad de las verificaciones y reproduce las
tres tendencias teóricas comprobadas.

---

### Siguiente paso del flujo de trabajo

**Etapa 3 — Preprocesamiento de datos:** limpieza y tratamiento de valores atípicos, división
estratificada en entrenamiento y prueba, escalado y selección de características, y SMOTE solo si
resultara necesario.

Cuaderno siguiente: `03_preprocesamiento.ipynb`